In [1]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from IPython.display import clear_output

### Данные

In [2]:
aneks = []

with open('/kaggle/input/datasets/artembez/anekdotes-data/anek.txt', 'r', encoding = 'utf-8') as f:
    aneks.extend(f.read().split('\n'))

In [3]:
len(aneks)

125103

In [4]:
class Vocabulary:
    def __init__(self):
        self.vocabulary = ['_', '#', '<'] # start, end, padding
        self.vocabulary.extend('абвгдеёжзийклмнопрстуфхцчшщъыьэюя')
        self.vocabulary.extend('абвгдеёжзийклмнопрстуфхцчшщъыьэюя'.upper())
        self.vocabulary.extend('0123456789 ,.!?-')

        self._idx2char = {i: char for i, char in enumerate(self.vocabulary)}
        self._char2idx = {char: i for i, char in enumerate(self.vocabulary)}

    def get_vocabulary(self):
        return self.vocabulary

    def idx2char(self, idx: int):
        if idx not in self._idx2char:
            return '<'
        return self._idx2char[idx]

    def char2idx(self, char: str):
        if char not in self._char2idx:
            return self.get_pad()
        return self._char2idx[char]

    def encode(self, text):
        result = [self.char2idx(char) for char in text]
        result = [self.get_sos()] + result + [self.get_eos()]
        return result

    def get_sos(self):
        return self.char2idx('_')

    def get_eos(self):
        return self.char2idx('#')

    def get_pad(self):
        return self.char2idx('<')
    

In [5]:
from torch.utils.data import Dataset

class Anekdotes(Dataset):
    def __init__(self, aneks):
        self.aneks = aneks
        self.vocab = Vocabulary()

    def __getitem__(self, idx):
        return torch.LongTensor(self.vocab.encode(self.aneks[idx]))

    def __len__(self):
        return len(self.aneks)

In [6]:
dataset = Anekdotes(aneks)

In [7]:
dataset[10]

tensor([ 0, 61, 22, 12, 20, 15, 12, 26, 79, 22, 18, 19, 12, 15, 79,  4, 23, 20,
        10, 23, 13, 14, 23, 81, 79, 55, 18, 15, 32, 14, 18, 79, 27,  8, 20,  8,
        11, 79,  7,  5,  3, 79, 27,  3, 21,  3, 79,  8, 16, 23, 79, 23,  7,  3,
        15, 18, 21, 32, 79,  8,  8, 79, 23, 22, 18, 19, 12, 22, 32, 81,  1])

In [8]:
aneks[10]

'Штирлиц топил буржуйку. Только через два часа ему удалось ее утопить.'

In [10]:
dataset.vocab.get_pad()

2

In [11]:
len(dataset.vocab.get_vocabulary())

85

### Сохраним все данные о закодированных символах в файл

In [13]:
import json

In [27]:
vocab = dataset.vocab.get_vocabulary()
vocab_info = {'sos': dataset.vocab.get_sos(), 
              'eos': dataset.vocab.get_eos(), 
              'pad': dataset.vocab.get_pad(),
              'vocab_len': len(dataset.vocab.get_vocabulary())}
vocab_idxs = {}
for char in vocab:
    vocab_idxs[char] = dataset.vocab.char2idx(char)
vocab_info['vocab_idxs'] = vocab_idxs

In [28]:
with open('vocab_info.json', 'w', encoding = 'utf-8') as f:
    json.dump(vocab_info, f)

In [ ]:
from torch.utils.data import random_split

train_dataset, valid_dataset = random_split(dataset, (int(len(dataset) * 0.9), len(dataset) - int(len(dataset) * 0.9)))

In [ ]:
from torch.nn.utils.rnn import pad_sequence
pad_idx = dataset.vocab.get_pad()

def collate_fn(batch):
    return pad_sequence([b[:256] for b in batch], padding_value = pad_idx, batch_first = True)

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size = 256, collate_fn = collate_fn, pin_memory = True, num_workers = 4)
valid_loader = DataLoader(valid_dataset, batch_size = 256, collate_fn = collate_fn, pin_memory = True, num_workers = 4)

### RNN

### [RNN](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html)

$$h_t = \tanh\left( x_tW_{xh}^T + b_{xh} + h_{t-1}W_{hh}^T + b_{hh} \right)$$

либо если `bias=False`, то

$$h_t = \tanh\left( x_tW_{xh}^T + h_{t-1}W_{hh}^T \right)$$

Параметры:

 - `input_size` - размерность $x_i$
 - `hidden_size` - размерность $h_i$
 - `num_layers` - количество слоев (по умолчанию 1)
 - `dropout` - какой делать дропаут при количестве слоев > 1 (по умолчанию 0)
 - `bidirectional` - двунаправленная

In [ ]:
layer = nn.RNN(input_size = 3, hidden_size = 6)

In [ ]:
layer

In [ ]:
print(layer.weight_hh_l0) # веса h нулевого слоя
print()
print(layer.weight_hh_l0.shape)

In [ ]:
print(layer.weight_ih_l0)
print()
print(layer.weight_ih_l0.shape)

In [ ]:
print(layer.bias_hh_l0)
print()
print(layer.bias_hh_l0.shape)

In [ ]:
print(layer.bias_ih_l0)
print()
print(layer.bias_ih_l0.shape)

In [ ]:
batch_size = 5
seq_len = 7
input_dim = 3

layer = nn.RNN(input_size = 3, hidden_size = 6, batch_first = True)

x = torch.randn(batch_size, seq_len, input_dim)

hs, h_last = layer(x)

print(hs)
print()
print(hs.shape)


print(h_last)
print()
print(h_last.shape)

### GRU

### [GRU](https://pytorch.org/docs/stable/generated/torch.nn.GRU.html)

$$r_t = \sigma\left( x_t W_{xr}^T + b_{xr} + h_{t-1}W_{hr}^T + b_{hr} \right)$$
$$z_t = \sigma\left( x_t W_{xz}^T + b_{xz} + h_{t-1}W_{hz}^T + b_{hz} \right)$$
$$\hat{h_t} = \tanh\left( x_tW_{xh}^T + b_{xh} + r_t * (h_{t-1}W_{hh}^T + b_{hh}) \right)$$
$$h_t = (1 - z_t) * \hat{h_t} + z_t * h_{t-1}$$

Параметры такие же как у RNN.

In [ ]:
layer = nn.GRU(input_size = 3, hidden_size = 6)
layer

In [ ]:
print(layer.weight_hh_l0)
print()
print(layer.weight_hh_l0.shape)

In [ ]:
print(layer.weight_ih_l0)
print()
print(layer.weight_ih_l0.shape)

In [ ]:
print(layer.bias_hh_l0)
print()
print(layer.bias_hh_l0.shape)

In [ ]:
print(layer.bias_ih_l0)
print()
print(layer.bias_ih_l0.shape)

### LSTM

### [LSTM](https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

Более страшные формулы, все в целом очень похоже на GRU. Параметры те же.

In [ ]:
layer = nn.LSTM(input_size = 3, hidden_size = 6)
layer

### AnekdotesRNN

In [ ]:
class AnekdotesRNN(nn.Module):
    def __init__(self, num_tokens: int, emb_size: int = 256, hidden_size : int = 256):
        super().__init__()
        # каждый токен кодируем вектором размерности emb_size
        # padding_idx - индекс нулевого слова, обновляться не будет
        self.embedding = nn.Embedding(num_tokens, emb_size, padding_idx = pad_idx)

        self.rnn = nn.LSTM(
            input_size = emb_size,
            hidden_size = hidden_size,
            num_layers = 3,
            dropout = 0.3,
            batch_first = True
        )

        self.output = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LeakyReLU(),
            nn.Linear(hidden_size, num_tokens)
        )

    def forward(self, x):
        x = self.embedding(x)
        x, _ = self.rnn(x)
        x = self.output(x)

        return x

In [ ]:
model = AnekdotesRNN(num_tokens = len(dataset.vocab.get_vocabulary()))

### Функции для обучения
в качестве таргета мы берем последовательность, начиная с 1 элемента, потому что предсказываем следующий токен
вход на обучение: [1, 2, 3, 4]  
таргет: [2, 3, 4, 5]

In [ ]:
from tqdm import tqdm

def train(model):
    model.train()

    train_loss = 0
    
    for x in tqdm(train_loader, desc = 'Train'):
        x = x.to(device)
        
        optimizer.zero_grad()

        output = model(x[:, :-1]).transpose(1, 2)
        
        loss = loss_fn(output, x[:, 1:])

        train_loss += loss.item()

        loss.backward()
        optimizer.step()

    train_loss /= len(train_loader)

    return train_loss

Поскольку мы используем `nn.CrossEntropyLoss`, то посмотрим в каком размере она ожидает входы:

 - Input: Shape $(C)$, $(N, C)$ or $(N, C, d_1, d_2, ..., d_K)$ with $K \geq 1$ in the case of $K$-dimensional loss.
 - Target: $(N)$ or $(N, d_1, d_2, ..., d_K)$ with $K \geq 1$ in the case of $K$-dimensional loss where each value should be between $[0, C)$.

In [ ]:
x = next(iter(train_loader))
output = model(x[:, :-1])
print(output.shape)
print(x[:, 1:].shape)

print()

print(output.transpose(1, 2).shape)

In [ ]:
print(len(dataset.vocab.get_vocabulary()))

In [ ]:
@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    total_loss = 0
    for x in tqdm(loader, desc = 'Evaluation'):
        x = x.to(device)

        output = model(x[:, :-1]).transpose(1, 2)

        loss = loss_fn(output, x[:, 1:])

        total_loss += loss.item()

    total_loss /= len(loader)
    return total_loss

In [ ]:
def plot_stats(
    train_loss: list[float],
    valid_loss: list[float],
    title
):
    plt.figure(figsize = (16, 10))
    plt.title(title)
    plt.plot(train_loss, label = 'Train loss')
    plt.plot(valid_loss, label = 'Valid loss')
    plt.legend()
    plt.grid()
    plt.show()

In [ ]:
def whole_train_value_cycle(
    model,
    title: str,
    num_epochs = 3,
):
    train_loss_history = []
    valid_loss_history = []

    for epoch in range(num_epochs):
        train_loss = train(model)
        valid_loss = evaluate(model, valid_loader)

        train_loss_history.append(train_loss)
        valid_loss_history.append(valid_loss)

        clear_output(wait = True)

        plot_stats(train_loss_history, valid_loss_history, title)

### Обучение

In [ ]:
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(device)
print(torch.cuda.get_device_name() if torch.cuda.is_available() else None)

In [ ]:
from torch.optim import Adam
model = model.to(device)

optimizer = Adam(model.parameters(), lr = 2e-3)
loss_fn = nn.CrossEntropyLoss(ignore_index = pad_idx)

whole_train_value_cycle(model, 'Anekdotes RNN', 10)

### Генерация анекдотов

In [ ]:
eos = dataset.vocab.get_eos()

def ids2string(ids):
    result = []

    for _ in ids:
        if _ != eos:
            result.append(dataset.vocab.idx2char(_))
    return "".join(result)

def batch2str ing(ids, prefix):
    ans = ""
    for i, substr in enumerate(ids):
        ans += prefix + ids2string(substr.tolist())

        if i != len(ids) - 1:
            ans += '\n'
    return ans

def pick_by_distributions(logits):
    return torch.distributions.Categorical(logits = logits).sample()
    #probs = torch.exp(logits)
    #sample = torch.distributions.Categorical(probs).sample()
    #return sample

@torch.inference_mode()
def get_continuation(model, prefix: str = '', max_len: int = 1000, count = 10, temperature: float = 1):
    x = torch.LongTensor([dataset.vocab.encode(prefix)[:-1]] * count).to(device)
    model.eval()

    eos_idx = dataset.vocab.get_eos()
    finished = torch.zeros(count, dtype = torch.bool, device = device)
    results = [[] for _ in range(count)]
    
    logits = model(x)[:, -1, :]
    outs = pick_by_distributions(logits / temperature).unsqueeze(1)

    for i in range(max_len):
        x = torch.cat([x, outs], dim = 1)

        for j in range(count):
            if not finished[j]:
                tok = outs[j, 0].item()
                if tok == eos_idx:
                    finished[j] = True
                else:
                    results[j].append(tok)
        if finished.all():
            break
        logits = model(x)[:, -1, :]
        outs = pick_by_distributions(logits / temperature).unsqueeze(1)
    #print(batch2string(x[:, len(prefix) + 1:], prefix + '|' ))
    for i, toks in enumerate(results):
        line = prefix + '|' + ''.join(dataset.vocab.idx2char(t) for t in toks)
        print(line)

In [ ]:
get_continuation(model, prefix = 'Приходит как-то', temperature = 0.2)

In [ ]:
get_continuation(model, prefix = 'Штирлиц', temperature = 0.7)

In [ ]:
for idx in [10, 150, 772, 1835]:
    print(aneks[idx])
    print()
    get_continuation(model, prefix = ' '.join(aneks[idx].split()[:5]), temperature = 0.5)
    print()
    print('-' * 60)
    print()

In [ ]:
def save_model(model):
    torch.save(model.state_dict(), 'model_jokes_weights.pth')

In [ ]:
save_model(model)